# 🦷 DII — Treino YOLOv8 com GPU (Google Colab)

**Antes de rodar:** Ative a GPU em `Ambiente de execução → Alterar tipo de hardware → T4 GPU`

Tempo estimado com T4 GPU:
- `yolov8n`: ~40 min (50 épocas)
- `yolov8s`: ~1.5h (50 épocas) ← recomendado
- `yolov8m`: ~3h (50 épocas)


In [ ]:
# ── CONFIGURAÇÕES — edite aqui ──────────────────────────────
ROBOFLOW_API_KEY = "SUA_API_KEY_AQUI"   # app.roboflow.com → Settings → API Key
ARCHITECTURE     = "yolov8s"            # yolov8n | yolov8s | yolov8m
EPOCHS           = 50
IMGSZ            = 640
BATCH            = 16                   # GPU aguenta batch maior

# Dataset Roboflow (Dental Implants 2.0)
RF_WORKSPACE = "yosafat_chandra05-yahoo-com"
RF_PROJECT   = "dental-implants-2.0"
RF_VERSION   = 1
# ────────────────────────────────────────────────────────────

In [ ]:
# Verificar GPU
import torch
print(f'GPU disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('AVISO: GPU não encontrada! Vá em Ambiente de execução → Alterar tipo de hardware → T4 GPU')

In [ ]:
# Instalar dependências
!pip install ultralytics roboflow -q
print('Instalação concluída!')

In [ ]:
# Baixar dataset do Roboflow
from roboflow import Roboflow
import os

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download("yolov8", location="/content/dataset")

YAML_PATH = dataset.location + "/data.yaml"
print(f'\nDataset baixado em: {dataset.location}')
print(f'data.yaml: {YAML_PATH}')
!cat {YAML_PATH}

In [ ]:
# Treinar
from ultralytics import YOLO
import time

print(f'Iniciando treino: {ARCHITECTURE} | {EPOCHS} épocas | batch={BATCH} | imgsz={IMGSZ}')
start = time.time()

model = YOLO(f"{ARCHITECTURE}.pt")
results = model.train(
    data=YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,          # GPU
    project='/content/runs',
    name='dii_train',
    patience=15,
    plots=True,
    exist_ok=True,
    amp=True,           # FP16 — mais rápido na GPU
    cache=True,         # Cache imagens na RAM
    workers=2,
)

elapsed = time.time() - start
print(f'\nTreino concluído em {elapsed/3600:.1f}h')

In [ ]:
# Ver métricas finais e gráficos
import os
from IPython.display import Image, display

run_dir = '/content/runs/dii_train'
best_pt = os.path.join(run_dir, 'weights', 'best.pt')

print('=== Métricas finais ===')
try:
    metrics = results.results_dict
    map50   = metrics.get('metrics/mAP50(B)', 0)
    map95   = metrics.get('metrics/mAP50-95(B)', 0)
    prec    = metrics.get('metrics/precision(B)', 0)
    recall  = metrics.get('metrics/recall(B)', 0)
    print(f'  mAP50:     {map50*100:.1f}%')
    print(f'  mAP50-95:  {map95*100:.1f}%')
    print(f'  Precisão:  {prec*100:.1f}%')
    print(f'  Recall:    {recall*100:.1f}%')
except Exception as e:
    print(f'  Erro ao ler métricas: {e}')

print(f'\nModelo salvo em: {best_pt}')
print(f'Tamanho: {os.path.getsize(best_pt)/1024/1024:.1f} MB')

# Mostrar gráfico de resultados
results_png = os.path.join(run_dir, 'results.png')
if os.path.exists(results_png):
    display(Image(results_png, width=900))

In [ ]:
# Baixar best.pt para o computador
from google.colab import files
import shutil

best_pt = '/content/runs/dii_train/weights/best.pt'

# Renomear com nome descritivo
import datetime
date_str  = datetime.datetime.now().strftime('%Y%m%d')
dest_name = f'dii_{ARCHITECTURE}_{EPOCHS}ep_{date_str}.pt'
shutil.copy(best_pt, f'/content/{dest_name}')

print(f'Baixando: {dest_name}')
files.download(f'/content/{dest_name}')
print('\nAgora faça upload no servidor DII:')
print('  1. No painel DII: Modelos → Fazer Upload de Modelo')
print(f'  2. Ou via scp: scp {dest_name} root@sandre.dev:/var/www/vhosts/sandre.dev/httpdocs/dii/models/')

In [ ]:
# OPCIONAL: Enviar direto para o servidor via API
# Preencha a URL e token do seu servidor DII

DII_URL   = "https://dii.sandre.dev"   # URL do seu servidor
DII_TOKEN = ""                          # Token JWT (faça login e copie do localStorage)

if DII_TOKEN:
    import requests

    best_pt   = '/content/runs/dii_train/weights/best.pt'
    map50_val = results.results_dict.get('metrics/mAP50(B)', 0)
    map95_val = results.results_dict.get('metrics/mAP50-95(B)', 0)
    prec_val  = results.results_dict.get('metrics/precision(B)', 0)
    rec_val   = results.results_dict.get('metrics/recall(B)', 0)

    # Upload do arquivo
    with open(best_pt, 'rb') as f:
        resp = requests.post(
            f'{DII_URL}/api/models/upload',
            headers={'Authorization': f'Bearer {DII_TOKEN}'},
            files={'model': (dest_name, f, 'application/octet-stream')},
            data={
                'name':         f'DII-Colab-{ARCHITECTURE}',
                'architecture': ARCHITECTURE,
                'epochs':       EPOCHS,
                'map50':        map50_val,
                'map95':        map95_val,
                'precision':    prec_val,
                'recall':       rec_val,
            }
        )

    if resp.status_code == 201:
        data = resp.json()
        print(f'Modelo enviado ao servidor!')
        print(f'Model ID: {data["model_id"]}')
        print(f'Acesse o painel DII e clique em Deploy para ativar.')
    else:
        print(f'Erro {resp.status_code}: {resp.text}')
else:
    print('Token não preenchido. Use a célula anterior para baixar o .pt manualmente.')